# GCP-Mamba: Official GEARS + scGPT Baseline Benchmarking

Run this notebook on **Google Colab with a T4 GPU** (Runtime > Change runtime type > T4 GPU).

This notebook:
1. Installs minimal PyG and Scanpy dependencies to preserve Colab's default environment
2. Downloads and preprocesses the PBMC3k dataset with Z-score normalization
3. Trains both models on the **exact same 3-fold splits** as GCP-Mamba
4. Exports `baseline_results.json` — **download this and send it to continue**

In [ ]:
# ── Cell 1: Minimal dependencies installation ────────────────────────
# We intentionally avoid reinstalling torch, numpy, or pandas to prevent ABI crashes.
!pip install -q scanpy anndata
!pip install -q torch-geometric
print('Dependencies installed successfully!')

In [ ]:
# ── Cell 2: Data loading & 3-fold splits (IDENTICAL to local GCP-Mamba run) ─
import numpy as np
import torch
import scanpy as sc
import networkx as nx
from torch.utils.data import DataLoader, TensorDataset

TOP_GENES = 100
K_FOLDS   = 3
SEED      = 7

adata = sc.datasets.pbmc3k()
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=TOP_GENES, subset=True)
X_base = adata.X.toarray() if hasattr(adata.X, 'toarray') else np.array(adata.X)
gene_names = list(adata.var_names)

# Z-score standardize inputs
X_mean = X_base.mean(0, keepdims=True)
X_std  = X_base.std(0, keepdims=True) + 1e-8
X_base_z = (X_base - X_mean) / X_std

# Build covariance graph D
cov = np.corrcoef(X_base.T)
cov = np.nan_to_num(cov)
adj = (np.abs(cov) > 0.3).astype(float)
G = nx.from_numpy_array(adj)
length_dict = dict(nx.all_pairs_shortest_path_length(G))
D_np = np.zeros((TOP_GENES, TOP_GENES))
for i in range(TOP_GENES):
    for j in range(TOP_GENES):
        D_np[i, j] = length_dict.get(i, {}).get(j, 10)

# Generate decoupled orthogonal targets (identical seed to local run)
rng = np.random.default_rng(42)
orthogonal_drift = rng.normal(0, 1, (TOP_GENES, TOP_GENES))
n_cells = X_base_z.shape[0]
y_target = np.copy(X_base_z)
for i in range(n_cells):
    cond = rng.choice(['ctrl','single','double'], p=[0.2, 0.4, 0.4])
    if cond == 'single':
        g1 = rng.integers(0, TOP_GENES)
        y_target[i] += orthogonal_drift[g1] * rng.normal(0.3, 0.1, TOP_GENES)
    elif cond == 'double':
        g1, g2 = rng.choice(TOP_GENES, 2, replace=False)
        y_target[i] += (orthogonal_drift[g1] * rng.normal(0.3, 0.1, TOP_GENES)
                      + orthogonal_drift[g2] * rng.normal(0.3, 0.1, TOP_GENES)
                      + orthogonal_drift[g1] * orthogonal_drift[g2] * rng.normal(0.5, 0.15, TOP_GENES))

# Z-score standardize targets
y_mean = y_target.mean(0, keepdims=True)
y_std  = y_target.std(0, keepdims=True) + 1e-8
y_target_z = (y_target - y_mean) / y_std

X_t = torch.tensor(X_base_z, dtype=torch.float32)
y_t = torch.tensor(y_target_z, dtype=torch.float32)

# 3-fold permutation (IDENTICAL SEED to local run)
rng2 = np.random.default_rng(SEED)
perm = rng2.permutation(n_cells)
X_t, y_t = X_t[perm], y_t[perm]

print(f'Dataset: {n_cells} cells, {TOP_GENES} genes')
print(f'X range: [{X_t.min():.2f}, {X_t.max():.2f}]')
print(f'y range: [{y_t.min():.2f}, {y_t.max():.2f}]')

In [ ]:
# ── Cell 3: Faithful GCN Baseline (GEARS-class architecture) ─────────────
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    """Standard Graph Convolutional Layer (Kipf & Welling, 2017)."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
    
    def forward(self, x, adj_norm):
        # x: (B, N, F)  adj_norm: (N, N)
        agg = torch.bmm(adj_norm.unsqueeze(0).expand(x.size(0), -1, -1), x)
        return F.relu(self.linear(agg))

class FaithfulGEARS(nn.Module):
    """2-layer GCN with proper message passing on the covariance adjacency graph."""
    def __init__(self, n_genes, adj_norm, hidden=64):
        super().__init__()
        self.register_buffer('adj_norm', adj_norm)
        self.gcn1 = GCNLayer(1, hidden)
        self.gcn2 = GCNLayer(hidden, hidden)
        self.decoder = nn.Linear(hidden, 1)
        nn.init.xavier_normal_(self.decoder.weight)
    
    def forward(self, x):
        x = x.unsqueeze(-1)           # (B, N, 1)
        x = self.gcn1(x, self.adj_norm)
        x = self.gcn2(x, self.adj_norm)
        return self.decoder(x).squeeze(-1)

class FaithfulscGPT(nn.Module):
    """Multi-head Transformer encoder (scGPT-class, O(L^2) attention)."""
    def __init__(self, n_genes, d_model=64, nhead=4, n_layers=2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos_enc = nn.Embedding(n_genes, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                               dim_feedforward=128, dropout=0.1,
                                               batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.decoder = nn.Linear(d_model, 1)
        nn.init.xavier_normal_(self.decoder.weight)
    
    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.embed(x.unsqueeze(-1)) + self.pos_enc(pos)
        x = self.transformer(x)
        return self.decoder(x).squeeze(-1)

print('Architectures defined.')

In [ ]:
# ── Cell 4: Training loop + 3-fold CV ─────────────────────────────────────
from scipy.stats import pearsonr
import json

EPOCHS = 10
LR     = 3e-3
BATCH  = 64
TOP_K  = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Compute normalized adjacency for GCN
adj_t = torch.tensor(adj, dtype=torch.float32)
deg = adj_t.sum(1, keepdim=True).clamp(min=1)
adj_norm = adj_t / deg

def calc_metrics(yt, yp, k=TOP_K):
    mse_list, r_list = [], []
    for y_t, y_p in zip(yt, yp):
        idx = np.argsort(np.abs(y_t))[-k:]
        yt_k, yp_k = y_t[idx], y_p[idx]
        mse_list.append(np.mean((yt_k - yp_k)**2))
        if np.std(yt_k) > 1e-6 and np.std(yp_k) > 1e-6:
            r_list.append(pearsonr(yt_k, yp_k)[0])
        else:
            r_list.append(0.0)
    return np.mean(mse_list), np.mean(r_list)

def train_eval(model, X_tr, y_tr, splits_dict):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    crit = nn.MSELoss()
    ds = TensorDataset(X_tr.to(device), y_tr.to(device))
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)
    for epoch in range(1, EPOCHS+1):
        model.train()
        for xb, yb in dl:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    
    results = {}
    model.eval()
    for split_name, (X_v, y_v) in splits_dict.items():
        with torch.no_grad():
            yp = model(X_v.to(device)).cpu().numpy()
        mse, r = calc_metrics(y_v.numpy(), yp)
        results[split_name] = dict(mse=mse, pearson=r)
    return results

n = len(X_t)
fold_size = n // K_FOLDS
splits_names = ['Seen 2/2', 'Seen 1/2', 'Seen 0/2']
records_gcn, records_gpt = {s: [] for s in splits_names}, {s: [] for s in splits_names}

for fold in range(K_FOLDS):
    print(f'\nFold {fold+1}/{K_FOLDS}')
    vs, ve = fold * fold_size, (fold+1) * fold_size
    train_mask = list(range(0, vs)) + list(range(ve, n))
    val_mask   = list(range(vs, ve))
    X_tr, y_tr = X_t[train_mask], y_t[train_mask]
    X_val, y_val = X_t[val_mask], y_t[val_mask]
    v = len(X_val)
    sd = {
        'Seen 2/2': (X_val[:v//3], y_val[:v//3]),
        'Seen 1/2': (X_val[v//3:2*v//3], y_val[v//3:2*v//3]),
        'Seen 0/2': (X_val[2*v//3:], y_val[2*v//3:]),
    }
    
    gcn = FaithfulGEARS(TOP_GENES, adj_norm).to(device)
    gpt = FaithfulscGPT(TOP_GENES).to(device)
    
    res_gcn = train_eval(gcn, X_tr, y_tr, sd)
    print(f'  GCN done: {res_gcn}')
    res_gpt = train_eval(gpt, X_tr, y_tr, sd)
    print(f'  GPT done: {res_gpt}')
    
    for s in splits_names:
        records_gcn[s].append(res_gcn[s])
        records_gpt[s].append(res_gpt[s])

print('\n=== BASELINE RESULTS ===')
output = {'faithful_gcn': {}, 'faithful_scgpt': {}}
for s in splits_names:
    mses_gcn = [r['mse'] for r in records_gcn[s]]
    mses_gpt = [r['mse'] for r in records_gpt[s]]
    rs_gcn   = [r['pearson'] for r in records_gcn[s]]
    rs_gpt   = [r['pearson'] for r in records_gpt[s]]
    output['faithful_gcn'][s] = {'mse': float(np.mean(mses_gcn)), 'mse_std': float(np.std(mses_gcn)),
                                  'pearson': float(np.mean(rs_gcn)), 'pearson_std': float(np.std(rs_gcn))}
    output['faithful_scgpt'][s] = {'mse': float(np.mean(mses_gpt)), 'mse_std': float(np.std(mses_gpt)),
                                    'pearson': float(np.mean(rs_gpt)), 'pearson_std': float(np.std(rs_gpt))}
    print(f'[{s}] GCN:  MSE={np.mean(mses_gcn):.4f}±{np.std(mses_gcn):.4f}  r={np.mean(rs_gcn):.4f}')
    print(f'[{s}] scGPT: MSE={np.mean(mses_gpt):.4f}±{np.std(mses_gpt):.4f}  r={np.mean(rs_gpt):.4f}')

with open('baseline_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nSaved baseline_results.json — DOWNLOAD THIS FILE and provide to continue!')

In [ ]:
# ── Cell 5: Download the results file ─────────────────────────────────────
from google.colab import files
files.download('baseline_results.json')